# 01 – Exploring the transaction dataset
Goal: understand the data before training a model that predicts a transaction's category from its description.
Dataset: [mitulshah/transaction-categorization](https://huggingface.co/datasets/mitulshah/transaction-categorization) (4.5M rows, MIT license)

In [1]:
import pandas as pd
from pathlib import Path

path = Path("../data/transactions.parquet")
if not path.exists():
    df = pd.read_parquet("hf://datasets/mitulshah/transaction-categorization/default/train/0000.parquet")
    path.parent.mkdir(exist_ok=True)
    df.to_parquet(path)

df = pd.read_parquet(path)

print(df.shape)
print(df.dtypes)
df.head(10)

(4501043, 4)
transaction_description    str
category                   str
country                    str
currency                   str
dtype: object


,transaction_description,category,country,currency
0,Wage,Income,USA,USD
1,Arby's (Contactless),Food & Dining,AUSTRALIA,AUD
2,Occupational Therapy,Healthcare & Medical,USA,USD
3,Potbelly Store Branch,Food & Dining,UK,GBP
4,Amazon - AUSTRALIA,Shopping & Retail,AUSTRALIA,AUD
5,Dairy Queen,Food & Dining,CANADA,CAD
6,UCLA Medical #7731 - UK Center TXN176162,Healthcare & Medical,UK,GBP
7,Walgreens #8780,Healthcare & Medical,AUSTRALIA,AUD
8,Starlink,Utilities & Services,USA,USD
9,Cox,Utilities & Services,UK,GBP


In [2]:
# 1. Missing values
print(df.isnull().sum())

transaction_description    0
category                   0
country                    0
currency                   0
dtype: int64


In [3]:
# 2. Class balance
print(df["category"].value_counts(normalize=True).round(3))

category
Utilities & Services          0.1
Government & Legal            0.1
Financial Services            0.1
Income                        0.1
Charity & Donations           0.1
Shopping & Retail             0.1
Healthcare & Medical          0.1
Entertainment & Recreation    0.1
Transportation                0.1
Food & Dining                 0.1
Name: proportion, dtype: float64


In [4]:
# 3. Is currency just a copy of country?
print(pd.crosstab(df["country"], df["currency"]))

currency      AUD     CAD     GBP     INR     USD
country                                          
AUSTRALIA  901765       0       0       0       0
CANADA          0  898656       0       0       0
INDIA           0       0       0  901544       0
UK              0       0  899915       0       0
USA             0       0       0       0  899163


In [5]:
# 4. Is country random?
print(pd.crosstab(df["country"], df["category"], normalize="index").round(3))

category   Charity & Donations  Entertainment & Recreation  \
country                                                      
AUSTRALIA                0.100                         0.1   
CANADA                   0.100                         0.1   
INDIA                    0.100                         0.1   
UK                       0.101                         0.1   
USA                      0.100                         0.1   

category   Financial Services  Food & Dining  Government & Legal  \
country                                                            
AUSTRALIA               0.100          0.100               0.100   
CANADA                  0.100          0.100               0.100   
INDIA                   0.101          0.099               0.101   
UK                      0.100          0.099               0.101   
USA                     0.100          0.100               0.100   

category   Healthcare & Medical  Income  Shopping & Retail  Transportation  \
country   

In [6]:
# 5. How many unique descriptions?
n_unique = df["transaction_description"].nunique()
print(f"Unique descriptions: {n_unique:,} ({n_unique / len(df):.1%})")
print(df["transaction_description"].value_counts().head(20))

Unique descriptions: 1,387,044 (30.8%)
transaction_description
Payroll            5660
Salary             5421
Temple             5170
Social Security    5070
Frontier           4575
Post Office        4571
CVS                4570
Walgreens          4552
Rite Aid           4542
Target             4100
Subway             3972
Dividend           2864
Income             2808
Maternity Leave    2784
Rental Income      2781
Royalty            2758
Gig Work           2758
Reward             2755
SPCA               2744
Capital Gains      2732
Name: count, dtype: int64


### Summary of first phase of dataset analysis

- `Null Values` : No null values.
- `Class Balance` : Every category is exaclty 10%.
- `Currency` : It is just a copy of country, the model doesnt need it, we will drop the column.
- `Country` : Every country has the same category mix (~10% each), so knowing the country tells us nothing about the category.
- `Unique descriptions` : 1,387,044 out of 4.5M (30.8%).

### Decision so far 

1. `Model Parameters` : 
    - Input -> Descritpion (Only column that gives useful information).
    - Output -> Category.
    Example : Dairy Queen -> Food & Dining

2. `Training` : 
    - Evaluate twice:
        - Random split -> how well the model does on merchants it has already seen.
        - Merchant split (all rows of a merchant go to either train or test) -> how well it does on new merchants it has never seen.
    - The gap between the two scores shows how much the model relies on memorising merchant names.  

3. `Spendwise changes` : 
    Switch the existing categories Spendwise has, to the ones offered by the dataset.

## Fininding description templates

In [7]:
sample = df.sample(200_000, random_state=42)

# 1. 40 random descriptions: look for repeated extra bits around the store name
for d in sample["transaction_description"].sample(40, random_state=1):
    print(d)

Bus
Nordstrom Online - Weekend
Walmart Supercenter Shopping Center
Theme Park Store
Doctors Without Borders #4635 - USA
Oxfam - AUSTRALIA
Baja Fresh
Target (Mobile)
Red Cross - UK
Bank of America (Card)
Target - INDIA - Dinner Time
Neiman Marcus
Transfer
Religious Organization Branch - Dinner Time
Disney+ #9887
Religious Organization - Evening
Lowe's - Morning
UCSF Medical #2190 (App)
Best Buy
IKEA
DEA - CANADA Store Online
Family Dollar
Business WiFi #4624 Store
Tennis Court #5747
Safeway Pharmacy
Cinemark
Cox Online - Weekend
Anytime Fitness Online
Barneys New York - INDIA
Cane's (Online)
Hulu - UK Store Online
KFC
Vacation Pay
Dunkin' Donuts
Spotify #2659 Store Strip Mall
NSA #4905 Branch
Legal #5610
Walgreens - AUSTRALIA Store
Xfinity #6201
Oxfam Store


In [8]:
import re

SUFFIX_WORDS = (
    r"(store|online|branch|strip mall|shopping center|mall|center|airport|station|"
    r"downtown|hospital|campus|residential|business district)"
)

def clean_merchant(desc: str) -> str:
    """
    Extract the merchant name from a raw transaction description.

    Args:
        desc: The raw transaction description,
            e.g. "Spotify #2659 Store Strip Mall - UK".

    Returns:
        The cleaned merchant name, e.g. "spotify".
    """
    s = desc.lower()
    s = s.split(" - ")[0]                     # cuts text at every '-' gives a list of strings, keeps only first element
    s = re.sub(r"\(.*?\)", "", s)             # deletes anything inside brackets
    s = re.sub(r"#\d+", "", s)                # deletes # followed by number
    s = re.sub(r"txn\d+", "", s)              # deletes txt followed by numbers
    s = re.sub(r"\s+", " ", s).strip()        # "payroll  hospital" -> "payroll hospital"
    # remove suffix words at the end, repeatedly ("store center airport")
    while True:
        new = re.sub(rf"\s+{SUFFIX_WORDS}$", "", s).strip()
        if new == s:
            break
        s = new
    return s

In [9]:
for d in sample["transaction_description"].sample(40, random_state=1):
    print(f"{d:45} -> {clean_merchant(d)}")

Bus                                           -> bus
Nordstrom Online - Weekend                    -> nordstrom
Walmart Supercenter Shopping Center           -> walmart supercenter
Theme Park Store                              -> theme park
Doctors Without Borders #4635 - USA           -> doctors without borders
Oxfam - AUSTRALIA                             -> oxfam
Baja Fresh                                    -> baja fresh
Target (Mobile)                               -> target
Red Cross - UK                                -> red cross
Bank of America (Card)                        -> bank of america
Target - INDIA - Dinner Time                  -> target
Neiman Marcus                                 -> neiman marcus
Transfer                                      -> transfer
Religious Organization Branch - Dinner Time   -> religious organization
Disney+ #9887                                 -> disney+
Religious Organization - Evening              -> religious organization
Lowe's - Morn

In [10]:
sample["merchant"] = sample["transaction_description"].apply(clean_merchant)

print("Unique descriptions:", sample["transaction_description"].nunique())
print("Unique merchants:   ", sample["merchant"].nunique())
print(sample["merchant"].value_counts().sample(30, random_state=0))

Unique descriptions: 108139
Unique merchants:    455
merchant
alamo                   392
dividend                489
walmart                 413
kfc                     371
wealth management       465
clinic                  403
credit                  543
urologist               451
border patrol           424
chiropractor            433
chase                   460
cold stone creamery     383
lab                     440
phone company           463
donation                479
whataburger             388
directv                 453
temple                 1008
subway                  722
physical therapy        433
homeless shelter        481
paternity leave         517
yoga studio             419
bojangles               384
community bank          480
blimpie                 381
crunch fitness          411
maven                   417
police department       422
flixbus                 428
Name: count, dtype: int64


In [11]:
clean = sample[["transaction_description", "merchant", "category"]].reset_index(drop=True)
clean.head(10)

,transaction_description,merchant,category
0,Exxon - CANADA Store,exxon,Transportation
1,Dunkin' Donuts - USA Branch,dunkin' donuts,Food & Dining
2,Zoo - INDIA Store - Night,zoo,Entertainment & Recreation
3,Lyft - UK Branch,lyft,Transportation
4,Ulta #9601 Store (Cash),ulta,Shopping & Retail
5,SPCA - INDIA (Online),spca,Charity & Donations
6,Full-time - UK Center,full-time,Income
7,Chevron,chevron,Transportation
8,Humane Society,humane society,Charity & Donations
9,Stanford Health #8765 Center Residential,stanford health,Healthcare & Medical


### Ambiguous Merchants

In [12]:
merchant_categories = sample.groupby("merchant")["category"].agg(["nunique", lambda c: c.value_counts().to_dict()])
merchant_categories.columns = ["n_categories", "breakdown"]
ambiguous = merchant_categories[merchant_categories["n_categories"] > 1]
print(ambiguous)
print(f"Rows with an ambiguous merchant: {sample['merchant'].isin(ambiguous.index).mean():.2%}")

                 n_categories  \
merchant                        
cvs                         2   
frontier                    2   
post office                 2   
rite aid                    2   
social security             2   
target                      2   
walgreens                   2   

                                                         breakdown  
merchant                                                            
cvs              {'Shopping & Retail': 448, 'Healthcare & Medic...  
frontier         {'Utilities & Services': 436, 'Transportation'...  
post office      {'Government & Legal': 449, 'Shopping & Retail...  
rite aid         {'Shopping & Retail': 437, 'Healthcare & Medic...  
social security         {'Income': 528, 'Government & Legal': 447}  
target            {'Shopping & Retail': 407, 'Food & Dining': 363}  
walgreens        {'Healthcare & Medical': 455, 'Shopping & Reta...  
Rows with an ambiguous merchant: 3.00%


### Notes

- As we can see from our Sample dataset, 7 merchants appear in two categories (3% of rows), since the description alone cant seperate them the model will only guess for these merchants.

In [13]:
df["merchant"] = df["transaction_description"].apply(clean_merchant)
print("Merchants in full dataset:", df["merchant"].nunique())

Merchants in full dataset: 484


In [14]:
missing = set(df["merchant"]) - set(sample["merchant"])
print(len(missing))
print(df.loc[df["merchant"].isin(missing), "merchant"].value_counts())

29
merchant
dining out                      7
atm withdrawal                  3
coffee shop                     2
grocery shopping                1
gasoline refill                 1
restaurant dinner               1
paycheck deposit                1
phone bill payment              1
rent payment                    1
pharmacy purchase               1
mortgage payment                1
movie tickets                   1
gas refill                      1
snack purchase                  1
mobile phone bill payment       1
doctor's visit                  1
clothing shopping               1
internet bill payment           1
gym membership renewal          1
car wash                        1
dinner at italian restaurant    1
lunch at local cafe             1
online gaming subscription      1
utility bill payment            1
movie theater tickets           1
credit card payment             1
point of sale shopping          1
500.00                          1
earnings credit                 1
Na

In [15]:
n_rare = df["merchant"].isin(missing).sum()
print(n_rare, f"({n_rare / len(df):.4%})")

38 (0.0008%)


### Full Dataset vs Sample

- The `Full` dataset contains 29 more merchants than the `Sample` one.
- These merchants are extremely rare (38 rows, 0.0008% ), they are phrases not merchant names (ex. dinner at italian restaurant)

In [16]:
cols = ["transaction_description", "merchant", "category"]

# Sample (200k rows): for experimenting, fast
clean_sample = sample[cols].reset_index(drop=True)
clean_sample.to_parquet("../data/clean_sample.parquet")

# Full (4.5M rows): for training the final model at the end
clean_full = df[cols].reset_index(drop=True)
clean_full.to_parquet("../data/clean_full.parquet")

clean_sample.head(10)

,transaction_description,merchant,category
0,Exxon - CANADA Store,exxon,Transportation
1,Dunkin' Donuts - USA Branch,dunkin' donuts,Food & Dining
2,Zoo - INDIA Store - Night,zoo,Entertainment & Recreation
3,Lyft - UK Branch,lyft,Transportation
4,Ulta #9601 Store (Cash),ulta,Shopping & Retail
5,SPCA - INDIA (Online),spca,Charity & Donations
6,Full-time - UK Center,full-time,Income
7,Chevron,chevron,Transportation
8,Humane Society,humane society,Charity & Donations
9,Stanford Health #8765 Center Residential,stanford health,Healthcare & Medical
